In [ ]:
IS_TRAIN     = True    
IS_INFERENCE = True  

In [ ]:
from pathlib import Path

WORKING          = Path("/kaggle/working/")
INPUT            = Path("/kaggle/input/")

FOLDER_IMAGES    = WORKING / "Viet-Chart-VQA-images"

MODEL_PATH       = "/kaggle/input/models/bixunhong/intern-700/other/default/1/InternVL_2"

OUTPUT_DIR       = WORKING / "work_dirs/internvl_chat_v2_0/InternVL2_1B_lora_viet_chart_vqa"

MERGED_OUTPUT    = WORKING / "work_dirs/internvl_chat_v2_0/InternVL2_1B_lora_viet_chart_vqa_merged"


VI_CHART_DATASET_PATH = Path("/kaggle/input/datasets/maituananh511/dataset-chart-vqa/vi_chart_dataset")

VIETNAMESE_DATA_PATH = Path("/kaggle/input/datasets/maituananh511/data-vietnamese/Data Vietnamese")
VIETNAMESE_IMAGES_PATH = VIETNAMESE_DATA_PATH / "images"
VIETNAMESE_JSONL_PATH  = VIETNAMESE_DATA_PATH / "viet_chart_vqa.jsonl"

print("Cấu hình hoàn tất.")

In [ ]:
!pip install transformers==4.44.2 peft==0.11.1 --force-reinstall -q

In [ ]:

!pip install -U -q   accelerate datasets

In [ ]:
!pip install --upgrade deepspeed

In [ ]:
!pip install -q packaging ninja datasets timm einops deepspeed bitsandbytes decord gdown scipy

In [ ]:


!git clone https://github.com/OpenGVLab/InternVL.git {WORKING / 'InternVL'}

%cd {WORKING / 'InternVL'}
!pip install -q -e internvl_chat/

In [ ]:
from datasets import load_from_disk, concatenate_datasets, Dataset
import json
from PIL import Image
import pyarrow as pa

vi_chart_dataset = load_from_disk(str(VI_CHART_DATASET_PATH))
print("vi_chart_dataset:", vi_chart_dataset)

In [ ]:

def normalize_turn(turn):
    if isinstance(turn, str):
        return {'role': 'assistant', 'content': turn}
    if isinstance(turn, dict):
        role = turn.get('role') or turn.get('from', '')
        if role in ('human', 'user'):      role = 'user'
        elif role in ('gpt', 'assistant'): role = 'assistant'
        for rk in ('assistant', 'user', 'human', 'gpt'):
            if rk in turn and 'content' not in turn and 'role' not in turn:
                role = 'assistant' if rk in ('assistant', 'gpt') else 'user'
                return {'role': role, 'content': str(turn[rk])}
        content = str(turn.get('content') or turn.get('value', ''))
        return {'role': role, 'content': content}
    return {'role': 'assistant', 'content': str(turn)}


def align_conversations_schema(dataset, reference_dataset):
    ref_conv_type   = reference_dataset.data.schema.field('conversations').type
    ref_struct_type = ref_conv_type.value_type
    field_order     = [ref_struct_type.field(i).name for i in range(ref_struct_type.num_fields)]

    pa_table  = dataset.data.table
    conv_arr  = pa_table.column('conversations').combine_chunks()
    struct_arr = conv_arr.values

    arrays = [struct_arr.field(f) for f in field_order]
    fields = [pa.field(f, pa.string()) for f in field_order]
    new_struct = pa.StructArray.from_arrays(arrays, fields=fields)
    new_conv   = pa.ListArray.from_arrays(conv_arr.offsets, new_struct)

    idx       = pa_table.schema.get_field_index('conversations')
    new_table = pa_table.set_column(idx, 'conversations', new_conv)
    return Dataset(new_table)

In [ ]:
import os

vietnamese_records = []
with open(VIETNAMESE_JSONL_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            vietnamese_records.append(json.loads(line))

print(f"Vietnamese dataset: {len(vietnamese_records)} records loaded")

vn_rows = []
for record in vietnamese_records:
    img_path = VIETNAMESE_IMAGES_PATH / record['image']
    try:
        image = Image.open(img_path).convert('RGB')
    except Exception as e:
        print(f"[WARN] Không đọc được ảnh {img_path}: {e}")
        continue

    convs = [normalize_turn(t) for t in record['conversations']]
    pairs = [(convs[i], convs[i+1]) for i in range(0, len(convs) - 1, 2)]

    for idx, (q, a) in enumerate(pairs):
        record_id = record['id'] if len(pairs) == 1 else f"{record['id']}_q{idx}"
        vn_rows.append({
            'id': record_id,
            'image': image,
            'conversations': [q, a],
        })

print(f"Vietnamese rows: {len(vn_rows)}")

TEST_SIZE     = 200
vn_train_rows = vn_rows[:-TEST_SIZE]
vn_test_rows  = vn_rows[-TEST_SIZE:]

vi_vietnamese_train = Dataset.from_list(vn_train_rows)
vi_vietnamese_test  = Dataset.from_list(vn_test_rows)

vi_vietnamese_train = align_conversations_schema(vi_vietnamese_train, vi_chart_dataset['train'])
vi_vietnamese_test  = align_conversations_schema(vi_vietnamese_test,  vi_chart_dataset['test'])

vi_chart_30k = vi_chart_dataset['train'].shuffle(seed=42, keep_in_memory=True).select(range(30000))

merged_train = concatenate_datasets([vi_chart_30k, vi_vietnamese_train])
merged_test  = concatenate_datasets([vi_chart_dataset['test'], vi_vietnamese_test])

vi_chart_dataset['train'] = merged_train
vi_chart_dataset['test']  = merged_test

print(f"\nTổng train: {len(vi_chart_dataset['train'])}")
print(f"Tổng test : {len(vi_chart_dataset['test'])}")

In [ ]:
sample = vi_chart_dataset['train'][0]
print("ID:", sample['id'])
print("Image size:", sample['image'].size)
print("Conversations:", sample['conversations'])

In [ ]:
from pathlib import Path

MODEL_PATH = Path(MODEL_PATH) 

with open(MODEL_PATH / 'config.json') as f:
    cfg = json.load(f)

CONV_STYLE = cfg.get('template', 'internlm2-chat')
print(f"conv_style: {CONV_STYLE}")

In [ ]:
import numpy as np
import torch
import torchvision.transforms as T
from PIL import Image
from torchvision.transforms.functional import InterpolationMode
from transformers import AutoModel, AutoTokenizer

In [ ]:


IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)


def build_transform(input_size):
    return T.Compose([
        T.Lambda(lambda img: img.convert('RGB') if img.mode != 'RGB' else img),
        T.Resize((input_size, input_size), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    ])


def find_closest_aspect_ratio(aspect_ratio, target_ratios, width, height, image_size):
    best_ratio_diff = float('inf')
    best_ratio      = (1, 1)
    area            = width * height
    for ratio in target_ratios:
        target_ar   = ratio[0] / ratio[1]
        ratio_diff  = abs(aspect_ratio - target_ar)
        if ratio_diff < best_ratio_diff:
            best_ratio_diff = ratio_diff
            best_ratio      = ratio
        elif ratio_diff == best_ratio_diff:
            if area > 0.5 * image_size * image_size * ratio[0] * ratio[1]:
                best_ratio = ratio
    return best_ratio


def dynamic_preprocess(image, min_num=1, max_num=12, image_size=448, use_thumbnail=False):
    orig_width, orig_height = image.size
    aspect_ratio            = orig_width / orig_height
    target_ratios           = set(
        (i, j) for n in range(min_num, max_num + 1)
        for i in range(1, n + 1) for j in range(1, n + 1)
        if i * j <= max_num and i * j >= min_num
    )
    target_ratios = sorted(target_ratios, key=lambda x: x[0] * x[1])
    target_ar     = find_closest_aspect_ratio(
        aspect_ratio, target_ratios, orig_width, orig_height, image_size)
    target_w      = image_size * target_ar[0]
    target_h      = image_size * target_ar[1]
    blocks        = target_ar[0] * target_ar[1]
    resized_img   = image.resize((target_w, target_h))
    processed     = []
    for i in range(blocks):
        box = (
            (i % (target_w  // image_size)) * image_size,
            (i // (target_w // image_size)) * image_size,
            ((i % (target_w  // image_size)) + 1) * image_size,
            ((i // (target_w // image_size)) + 1) * image_size,
        )
        processed.append(resized_img.crop(box))
    assert len(processed) == blocks
    if use_thumbnail and len(processed) != 1:
        processed.append(image.resize((image_size, image_size)))
    return processed


def load_image(image_file, input_size=448, max_num=12):
    image = Image.open(image_file).convert('RGB') if isinstance(image_file, str) else image_file
    transform    = build_transform(input_size=input_size)
    images       = dynamic_preprocess(image, image_size=input_size, use_thumbnail=True, max_num=max_num)
    pixel_values = torch.stack([transform(img) for img in images])
    return pixel_values



In [ ]:
import matplotlib.pyplot as plt
import torch 

tokenizer      = AutoTokenizer.from_pretrained(str(MODEL_PATH), trust_remote_code=True, use_fast=False)
pretrain_model = AutoModel.from_pretrained(
    str(MODEL_PATH),
    torch_dtype=torch.float16,  
    low_cpu_mem_usage=True,
    trust_remote_code=True,
).eval().cuda()

generation_config = dict(max_new_tokens=512, do_sample=False, num_beams=3, repetition_penalty=2.0)

for i in range(min(3, len(vi_chart_dataset['train']))):
    item  = vi_chart_dataset['train'][i]
    image = item['image']
    convs = item['conversations']
    question_text = str(convs[0].get('content') or convs[0].get('value', ''))
    gt            = str(convs[1].get('content') or convs[1].get('value', ''))

    plt.figure(figsize=(8, 6))
    plt.imshow(image)
    plt.axis('off')
    plt.title(f"Sample {i}")
    plt.show()

  
    pixel_values = load_image(image, max_num=6).to(torch.float16).cuda()
    question     = f'<image>\n{question_text}'
    response     = pretrain_model.chat(tokenizer, pixel_values, question, generation_config)

    print(f"Q   : {question_text}")
    print(f"Pred: {response}")
    print(f"GT  : {gt}")
    print('=' * 60)

del pretrain_model
torch.cuda.empty_cache()

In [ ]:
from tqdm import tqdm

os.makedirs(FOLDER_IMAGES, exist_ok=True)

for i in tqdm(range(len(vi_chart_dataset['train'])), desc="Saving images"):
    item  = vi_chart_dataset['train'][i]
    image = item['image']
    id    = item['id']
    image.resize((448, 448)).save(f"{FOLDER_IMAGES}/{id}.jpg")

print(f" Đã lưu {len(vi_chart_dataset['train'])} ảnh vào {FOLDER_IMAGES}")

In [ ]:
def normalize_conversations(conversations):
    """Chuẩn hóa conversations về dạng [{'from': 'human'/'gpt', 'value': ...}]"""
    role_map = {'human': 'human', 'user': 'human', 'gpt': 'gpt', 'assistant': 'gpt'}
    result   = []
    for turn in conversations:
        raw_role = turn.get('from') or turn.get('role', '')
        role     = role_map.get(str(raw_role).lower(), raw_role)
        value    = str(turn.get('content') or turn.get('value', ''))
        result.append({'from': role, 'value': value})
    return result


all_data = []
for item in tqdm(vi_chart_dataset['train'], desc="Building JSONL"):
    id     = item['id']
    width  = item['image'].width
    height = item['image'].height
    normalized_convs = normalize_conversations(item['conversations'])

    for i in range(0, len(normalized_convs), 2):
        if i + 1 < len(normalized_convs):
            all_data.append({
                "id":    id,
                "image": f"{id}.jpg",
                "width": width,
                "height": height,
                "conversations": [
                    {"from": "human", "value": normalized_convs[i]['value']},
                    {"from": "gpt",   "value": normalized_convs[i + 1]['value']}
                ]
            })

print(f"Tổng mẫu training: {len(all_data)}")
print("\nVí dụ 2 mẫu đầu:")
print(json.dumps(all_data[:2], indent=2, ensure_ascii=False))

In [ ]:
JSONL_PATH = WORKING / 'InternVL/internvl_chat/shell/data/viet-chart-vqa.jsonl'
os.makedirs(JSONL_PATH.parent, exist_ok=True)

with open(JSONL_PATH, 'w', encoding='utf-8') as f:
    for data in tqdm(all_data, desc="Writing JSONL"):
        f.write(json.dumps(data, ensure_ascii=False) + '\n')

print(f" Đã lưu JSONL: {JSONL_PATH}")

META_PATH = WORKING / 'InternVL/internvl_chat/shell/data/custom_finetune_datasets.json'

metadata_datasets = {
    "vi-chart-vqa": {
        "root":         str(FOLDER_IMAGES),
        "annotation":   str(JSONL_PATH),
        "data_augment": False,
        "repeat_time":  1,
        "length":       len(vi_chart_dataset['train'])
    }
}

with open(META_PATH, 'w', encoding='utf-8') as f:
    json.dump(metadata_datasets, f, ensure_ascii=False, indent=4)

print(f"Đã lưu meta JSON: {META_PATH}")

In [ ]:
NUM_GPUS                  = 1    
PER_DEVICE_TRAIN_BATCH    = 1      
GRADIENT_ACC_STEPS        = 64   
NUM_TRAIN_EPOCHS          = 1      
LEARNING_RATE             = 2e-4   
LORA_RANK                 = 16     
MAX_SEQ_LENGTH            = 512   
MAX_DYNAMIC_PATCH         = 4    
SAVE_STEPS                = 500    
LOGGING_STEPS             = 10    
USE_BF16                  = False  
USE_FP16                  = True   

FREEZE_BACKBONE           = True   
FREEZE_MLP                = True   
FREEZE_LLM                = True   

print(f"Effective batch size: {NUM_GPUS} GPU × {PER_DEVICE_TRAIN_BATCH} × {GRADIENT_ACC_STEPS} acc = {NUM_GPUS * PER_DEVICE_TRAIN_BATCH * GRADIENT_ACC_STEPS}")

In [ ]:
TRAIN_SH_PATH = WORKING / 'InternVL/internvl_chat/shell/internvl2.0/2nd_finetune/internvl2_1b_lora_viet_chart_vqa.sh'
os.makedirs(TRAIN_SH_PATH.parent, exist_ok=True)

bf16_flag = 'False'
fp16_flag = 'True'

train_bash_content = f"""#!/bin/bash
set -x

export PYTHONPATH="$PYTHONPATH:$(pwd)"
export TF_CPP_MIN_LOG_LEVEL=3

OUTPUT_DIR="{OUTPUT_DIR}"
mkdir -p "$OUTPUT_DIR"

# DÙNG TORCHRUN ĐỂ CHẠY 2x T4 (Tự động cấp RANK/WORLD_SIZE)
torchrun --nproc_per_node=2 --master_port=29501 \\
  internvl/train/internvl_chat_finetune.py \\
  --model_name_or_path "{MODEL_PATH}" \\
  --conv_style "{CONV_STYLE}" \\
  --output_dir "$OUTPUT_DIR" \\
  --meta_path "{META_PATH}" \\
  --overwrite_output_dir True \\
  --force_image_size 336 \\
  --max_dynamic_patch 4 \\
  --down_sample_ratio 0.5 \\
  --drop_path_rate 0.0 \\
  --freeze_llm {FREEZE_LLM} \\
  --freeze_mlp {FREEZE_MLP} \\
  --freeze_backbone {FREEZE_BACKBONE} \\
  --use_llm_lora {LORA_RANK} \\
  --vision_select_layer -1 \\
  --dataloader_num_workers 2 \\
  --bf16 {bf16_flag} \\
  --fp16 {fp16_flag} \\
  --num_train_epochs {NUM_TRAIN_EPOCHS} \\
  --per_device_train_batch_size {PER_DEVICE_TRAIN_BATCH} \\
  --gradient_accumulation_steps {GRADIENT_ACC_STEPS} \\
  --evaluation_strategy no \\
  --save_strategy steps \\
  --save_steps {SAVE_STEPS} \\
  --save_total_limit 2 \\
  --learning_rate {LEARNING_RATE} \\
  --weight_decay 0.01 \\
  --warmup_ratio 0.03 \\
  --lr_scheduler_type "cosine" \\
  --logging_steps 10 \\
  --max_seq_length 512 \\
  --do_train True \\
  --grad_checkpointing True \\
  --group_by_length True \\
  --dynamic_image_size True \\
  --use_thumbnail True \\
  --ps_version v2 \\
  --launcher "pytorch" \\
  --report_to "none" \\
  2>&1 | tee -a "$OUTPUT_DIR/training_log.txt"
"""

with open(TRAIN_SH_PATH, 'w') as f:
    f.write(train_bash_content)

print(f"Script training đã FIX LỖI lưu tại: {TRAIN_SH_PATH}")

In [ ]:
!pip install -U "setuptools<70.0.0"

In [ ]:
!sed -i 's/import deepspeed/# import deepspeed/g' /kaggle/working/InternVL/internvl_chat/internvl/dist_utils.py

!sed -i 's/if deepspeed.checkpoint/# if False/g' /kaggle/working/InternVL/internvl_chat/internvl/dist_utils.py

print("Đã vô hiệu hóa DeepSpeed. Bây giờ script sẽ không bị crash khi khởi động nữa.")

In [ ]:
!pip install --no-cache-dir torchvision --index-url https://download.pytorch.org/whl/cu130 --force-reinstall -q



In [ ]:
import os

PATH_PATCH_DIR = "/kaggle/working/InternVL/internvl_chat/internvl/patch"
PATH_TRAIN_FILE = "/kaggle/working/InternVL/internvl_chat/internvl/train/internvl_chat_finetune.py"

fix_llama_patch = """
import warnings

try:
    from flash_attn.flash_attn_interface import flash_attn_varlen_func
    HAS_FLASH_ATTN = True
except ImportError:
    HAS_FLASH_ATTN = False

def replace_llama_attention_class():
    if not HAS_FLASH_ATTN:
        warnings.warn("FlashAttention không được cài đặt. Bỏ qua việc thay thế Llama Attention.")
        return
    print("FlashAttention detected. Patching Llama...")
"""
with open(os.path.join(PATH_PATCH_DIR, "llama_packed_training_patch.py"), "w") as f:
    f.write(fix_llama_patch)

fix_init = """
import warnings

def safe_import_patch(patch_name, func_name):
    try:
        module = __import__(f".{patch_name}", globals(), locals(), [func_name], 1)
        return getattr(module, func_name)
    except (ImportError, AttributeError, ModuleNotFoundError):
        # Trả về một hàm trống (no-op) nếu không import được
        return lambda: None

# Vô hiệu hóa các patch gây lỗi Flash Attention
replace_llama_attention_class = safe_import_patch("llama_packed_training_patch", "replace_llama_attention_class")
replace_llama2_attn_with_flash_attn = safe_import_patch("llama2_flash_attn_monkey_patch", "replace_llama2_attn_with_flash_attn")
replace_internlm2_attention_class = safe_import_patch("internlm2_packed_training_patch", "replace_internlm2_attention_class")
concat_pad_data_collator = safe_import_patch("pad_data_collator", "concat_pad_data_collator")
"""
with open(os.path.join(PATH_PATCH_DIR, "__init__.py"), "w") as f:
    f.write(fix_init)

with open(PATH_TRAIN_FILE, 'r') as f:
    lines = f.readlines()

with open(PATH_TRAIN_FILE, 'w') as f:
    for line in lines:
        if "replace_llama_attention_class()" in line or \
           "replace_llama2_attn_with_flash_attn()" in line or \
           "replace_internlm2_attention_class()" in line:
            f.write(f"    # {line.strip()} # Đã vô hiệu hóa để chạy trên T4\\n")
        else:
            f.write(line)


In [ ]:
import os

PATH_PATCH_INIT = "/kaggle/working/InternVL/internvl_chat/internvl/patch/__init__.py"

fix_init_v3 = """
import warnings

def safe_import_patch(patch_name, func_name):
    try:
        # Thử nạp module từ file tương ứng
        module = __import__(f".{patch_name}", globals(), locals(), [func_name], 1)
        return getattr(module, func_name)
    except (ImportError, AttributeError, ModuleNotFoundError):
        # Nếu lỗi (thiếu Flash Attention...), trả về một hàm rỗng để script không crash
        return lambda *args, **kwargs: None

# 1. Khai báo danh sách các hàm mà script Train (internvl_chat_finetune.py) yêu cầu
replace_llama_attention_class = safe_import_patch("llama_packed_training_patch", "replace_llama_attention_class")
replace_llama2_attn_with_flash_attn = safe_import_patch("llama2_flash_attn_monkey_patch", "replace_llama2_attn_with_flash_attn")
replace_llama_attn_with_flash_attn = safe_import_patch("llama_flash_attn_monkey_patch", "replace_llama_attn_with_flash_attn")
replace_internlm2_attention_class = safe_import_patch("internlm2_packed_training_patch", "replace_internlm2_attention_class")
replace_phi3_attention_class = safe_import_patch("phi3_packed_training_patch", "replace_phi3_attention_class")
replace_qwen2_attention_class = safe_import_patch("qwen2_packed_training_patch", "replace_qwen2_attention_class")

# 2. Các hàm tối ưu hóa hệ thống (RMSNorm, Dataloader)
# Hàm này gây lỗi ImportError bạn vừa gặp - giờ đã được khai báo an toàn
replace_llama_rmsnorm_with_fused_rmsnorm = safe_import_patch("llama_rmsnorm_monkey_patch", "replace_llama_rmsnorm_with_fused_rmsnorm")
replace_train_dataloader = safe_import_patch("train_dataloader_patch", "replace_train_dataloader")
replace_train_sampler = safe_import_patch("train_sampler_patch", "replace_train_sampler")

# 3. Xử lý Data Collator (Quan trọng để nạp dữ liệu)
try:
    from .pad_data_collator import concat_pad_data_collator, pad_data_collator, dpo_concat_pad_data_collator
except ImportError:
    from transformers import default_data_collator
    concat_pad_data_collator = default_data_collator
    pad_data_collator = default_data_collator
    dpo_concat_pad_data_collator = default_data_collator

# Xuất bản tất cả các tên hàm ra ngoài để script Train thấy được
__all__ = [
    'replace_llama_attn_with_flash_attn',
    'replace_llama_rmsnorm_with_fused_rmsnorm',
    'replace_llama2_attn_with_flash_attn',
    'replace_train_sampler',
    'replace_train_dataloader',
    'replace_internlm2_attention_class',
    'replace_qwen2_attention_class',
    'replace_phi3_attention_class',
    'replace_llama_attention_class',
    'concat_pad_data_collator'
]
"""

with open(PATH_PATCH_INIT, "w") as f:
    f.write(fix_init_v3)

In [ ]:
import builtins

In [ ]:
import os

file_path = "/kaggle/working/InternVL/internvl_chat/internvl/train/internvl_chat_finetune.py"

standard_huggingface_code = """
import os
os.environ['LD_LIBRARY_PATH'] = '/usr/local/cuda/lib64:' + os.environ.get('LD_LIBRARY_PATH', '')
os.environ["BNB_CUDA_VERSION"] = "121"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
import torch.distributed as dist
from dataclasses import dataclass, field
from typing import Optional
from transformers import (AutoTokenizer, HfArgumentParser, Trainer, TrainingArguments)
from internvl.model.internvl_chat import InternVLChatConfig, InternVLChatModel
from internvl.patch import (concat_pad_data_collator)
from internvl.train.dataset import (build_transform, preprocess, preprocess_internlm, preprocess_mpt)
from torch.utils.data import Dataset, ConcatDataset
import json
from copy import deepcopy

IMG_CONTEXT_TOKEN = '<IMG_CONTEXT>'

@dataclass
class ModelArguments:
    model_name_or_path: Optional[str] = field(default=None)
    use_llm_lora: int = field(default=0)
    freeze_llm: bool = field(default=True)
    freeze_mlp: bool = field(default=True)
    freeze_backbone: bool = field(default=True)
    vision_select_layer: int = field(default=-1)
    grad_checkpointing: bool = field(default=True) 
    launcher: str = field(default='pytorch')
    drop_path_rate: float = field(default=0.0)
    ps_version: str = field(default='v2')

@dataclass
class DataTrainingArguments:
    max_seq_length: int = field(default=512)
    force_image_size: int = field(default=336)
    down_sample_ratio: float = field(default=0.5) 
    max_dynamic_patch: int = field(default=1)
    meta_path: str = field(default=None)
    conv_style: str = field(default='internlm2-chat')
    dynamic_image_size: bool = field(default=False)
    use_thumbnail: bool = field(default=False)

class LazySupervisedDataset(Dataset):
    def __init__(self, template_name, meta, tokenizer, ds_name, num_image_token, image_size=336):
        super().__init__()
        self.tokenizer, self.template_name = tokenizer, template_name
        self.num_image_token = num_image_token
        self.root = meta['root']
        with open(meta['annotation'], 'r') as f: self.raw_data = f.readlines()
        self.image_size = image_size

    def __len__(self): return len(self.raw_data)
    def __getitem__(self, i):
        data_item = json.loads(self.raw_data[i])
        from PIL import Image
        image = Image.open(os.path.join(self.root, data_item['image'])).convert('RGB').resize((self.image_size, self.image_size))
        transform = build_transform(is_train=True, input_size=self.image_size)
        pixel_values = torch.stack([transform(image)])
        convs = deepcopy(data_item['conversations'])
        if '<image>' not in convs[0]['value']: convs[0]['value'] = '<image>\\\\n' + convs[0]['value']
        pre_func = preprocess_mpt if self.template_name == 'Hermes-2' else (preprocess_internlm if self.template_name == 'internlm2-chat' else preprocess)
        ret = pre_func(self.template_name, [convs], self.tokenizer, [self.num_image_token])
        return dict(input_ids=ret['input_ids'][0], labels=ret['labels'][0], attention_mask=ret['attention_mask'][0], pixel_values=pixel_values, image_flags=torch.tensor([1]))

def main():
    if 'RANK' in os.environ: dist.init_process_group(backend='nccl')
    else: dist.init_process_group(backend='nccl', rank=0, world_size=1)

    parser = HfArgumentParser((ModelArguments, DataTrainingArguments, TrainingArguments))
    model_args, data_args, training_args = parser.parse_args_into_dataclasses()

    tokenizer = AutoTokenizer.from_pretrained(model_args.model_name_or_path, trust_remote_code=True)
    tokenizer.model_max_length = data_args.max_seq_length
    tokenizer.add_tokens([IMG_CONTEXT_TOKEN], special_tokens=True)
    
    config = InternVLChatConfig.from_pretrained(model_args.model_name_or_path)
    if isinstance(config, tuple): config = config[0]
    
    if hasattr(config, 'vision_config'):
        config.vision_config.image_size = data_args.force_image_size
    
    config.max_seq_length = data_args.max_seq_length
    if hasattr(config, 'llm_config'):
        config.llm_config.max_position_embeddings = data_args.max_seq_length
        config.llm_config.model_max_length = data_args.max_seq_length
        config.llm_config._attn_implementation = 'eager'

    model = InternVLChatModel.from_pretrained(
        model_args.model_name_or_path, 
        torch_dtype=torch.float32, 
        config=config, 
        attn_implementation='eager',
        ignore_mismatched_sizes=True
    )
    
    model.language_model.resize_token_embeddings(len(tokenizer))
    model.img_context_token_id = tokenizer.convert_tokens_to_ids(IMG_CONTEXT_TOKEN)
    model.num_image_token = 144 

    ds_meta = json.loads(open(data_args.meta_path).read())
    datasets = [LazySupervisedDataset(data_args.conv_style, ds_meta[name], tokenizer, name, 144, image_size=data_args.force_image_size) for name in ds_meta.keys()]
    
    if model_args.use_llm_lora:
        model.wrap_llm_lora(r=model_args.use_llm_lora, lora_alpha=2 * model_args.use_llm_lora)

    model.gradient_checkpointing_enable()
    model.language_model.config.use_cache = False

    # Đã xóa sạch các đoạn hack "scaler = None"
    trainer = Trainer(model=model, args=training_args, train_dataset=ConcatDataset(datasets), data_collator=concat_pad_data_collator, tokenizer=tokenizer)
    import builtins
    original_print = builtins.print

    def filtered_print(*args, **kwargs):
        # Nếu nội dung in ra có chứa cụm từ spam thì bỏ qua không in
        if args and isinstance(args[0], str) and 'dynamic ViT batch size' in args[0]:
            return 
        # Còn lại thì in bình thường
        original_print(*args, **kwargs)

    # Ghi đè hàm print mặc định của Python
    builtins.print = filtered_print
    trainer.train()

if __name__ == '__main__': main()
"""

with open(file_path, "w") as f:
    f.write(standard_huggingface_code)

In [ ]:
import os

file_path = "/kaggle/working/InternVL/internvl_chat/internvl/train/internvl_chat_finetune.py"

if os.path.exists(file_path):
    with open(file_path, 'r') as f:
        lines = f.readlines()

    with open(file_path, 'w') as f:
        f.write("import os\n")
        f.write("os.environ['LD_LIBRARY_PATH'] = os.environ.get('LD_LIBRARY_PATH', '') + ':/usr/local/cuda/lib64'\n")
        
        for line in lines:
            if line.strip() == "import os": continue
            f.write(line)
            

In [ ]:
import os

!pip install -U bitsandbytes --quiet

os.environ['LD_LIBRARY_PATH'] = '/usr/local/cuda/lib64:' + os.environ.get('LD_LIBRARY_PATH', '')

os.environ["BNB_CUDA_VERSION"] = "121" 


In [ ]:
import os

!pip uninstall -y deepspeed
!pip install deepspeed --quiet

!pip install -U bitsandbytes --quiet
os.environ['LD_LIBRARY_PATH'] = '/usr/local/cuda/lib64:' + os.environ.get('LD_LIBRARY_PATH', '')
os.environ["BNB_CUDA_VERSION"] = "121"


In [ ]:
if IS_TRAIN:
    %cd {WORKING / 'InternVL/internvl_chat'}
    !sh {TRAIN_SH_PATH}
else:
    print("IS_TRAIN = False — bỏ qua bước training.")